# **LLM Call


In [3]:
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv, find_dotenv
import os

# Load environment variables from .env file
# find_dotenv() searches parent folders for the .env file
load_dotenv(find_dotenv())

# Initialize the model
model = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

# Invoke the model
response = model.invoke("Hello! Capitol Of UAE")
print(response.content)

# Capital of UAE

The capital of the **United Arab Emirates (UAE)** is **Abu Dhabi**.

## Key Facts:
- **Location**: Located on the southeast coast of the Persian Gulf
- **Largest Emirate**: Abu Dhabi is the largest and richest of the seven emirates
- **Population**: Over 1.5 million people
- **Notable Features**:
  - Sheikh Zayed Grand Mosque
  - Emirates Palace
  - Corniche waterfront
  - Modern skyline with impressive architecture

While **Dubai** is the most famous and visited emirate, Abu Dhabi is the political and administrative capital of the UAE.


In [4]:
from langchain.chat_models import init_chat_model

llm_claude = init_chat_model(model ="claude-haiku-4-5" )
llm_claude.invoke("Whats the probability of heads").content

"# Probability of Heads\n\nFor a fair coin flip, the probability of heads is **1/2 or 0.5 (50%)**.\n\nThis assumes:\n- The coin is fair (not weighted)\n- There are only two possible outcomes (heads or tails)\n- Each outcome is equally likely\n\nIs there a specific scenario you're asking about?"

# **Messages

In [5]:
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage 

my_messages = [ 
                SystemMessage(content="You are a character from Shakesperes plays"),
                HumanMessage(content="How to tell my girlfriend she is the prettiest girl in the whole world")
                ]

llm_claude.invoke(my_messages).content

'# A Shakespearean Declaration\n\nAh, good friend! Let me offer counsel in the manner of the Bard himself:\n\n**Speak with sincerity, not mere flattery:**\n- "Thou art to me the fairest in all the realm" carries more weight than hollow praise\n- Be specific—admire her eyes, her wit, her character—not just her appearance\n\n**Time your moment well:**\n- Choose a quiet, genuine moment, not forced or theatrical\n- Let it arise naturally in conversation\n\n**Use your own words:**\n- Authentic feeling matters more than eloquent speech\n- A simple "I think you\'re beautiful" from the heart beats any sonnet\n\n**But heed this warning** (as Hamlet might caution):\n- Beauty fades, but character endures\n- Tell her what else you admire—her strength, her humor, her kindness\n- Shallow compliments alone breed doubt\n\n**My final thought:**\nThe most moving declarations combine praise of her beauty *with* recognition of who she is within. That\'s how love is truly expressed—not as idolatry of form,

In [6]:
from langchain_core.prompts import PromptTemplate

user_input =input("enter  a topic for fun joke")
dynamic_prompt = PromptTemplate.from_template('Write a joke about {topic}')

ready_prompt = dynamic_prompt.invoke({"topic" : user_input})
llm_claude.invoke(ready_prompt).content

'# 🚴 Bike Joke\n\nWhy did the bicycle fall over?\n\nBecause it was **two-tired!**\n\n---\n\n*Bonus version:* A guy walks into a bike shop and asks, "Do you have anything to help me with my cycling?"\n\nThe shopkeeper replies, "Sure! We have some great bikes."\n\nThe guy says, "No, I meant my *drug problem.*"'

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template =ChatPromptTemplate.from_messages([
    ("system","You are an {tone} boss"),
    ("user", "Requesting a leave on the {dates}")
])


user_input = input("Which date do you want to apply for leave")
boss_tone = input("How should the boss react?")
ready_prompt = prompt_template.invoke({"dates":user_input,"tone":boss_tone})

llm_claude.invoke(ready_prompt.messages).content

"Thank you for reaching out. I'd be happy to help with your leave request for August 15th.\n\nTo process this properly, could you please provide me with a few details:\n\n1. **Type of leave** - Is this a vacation day, sick leave, personal day, or another type?\n2. **Duration** - Will this be a full day or partial day?\n3. **Reason** (optional) - If you're comfortable sharing, it helps with planning\n4. **Advance notice** - How much notice are you providing?\n\nOnce I have these details, I'll review your leave balance and get this approved for you. Please also ensure your work responsibilities are covered during your absence.\n\nLooking forward to hearing from you.\n\nBest regards"

# **Pydantic Models


In [ ]:
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    setup: str = Field(description="Setup for the joke")
    punchline: str = Field(description="Punchline for the joke")



In [22]:
obj =llm_schema(**{"setup": "something","punchline": "12"})
obj

llm_schema(setup='something', punchline='12')

In [26]:
llm_structured_output = llm_claude.with_structured_output(llm_schema)
result = llm_structured_output.invoke("Bar jokes")
result.punchline


'The bartender looks up and says, "Hey, we have a drink named after you!" The grasshopper looks confused and says, "What? You have a drink called Steve?"'

# **TypeDict


In [27]:
from typing import TypedDict

class llm_schema_td(TypedDict):
    setup: str
    punchline: str

In [32]:
llm_structured_typed_dict = llm_claude.with_structured_output(llm_schema_td)

result = llm_structured_typed_dict.invoke("tell me a joke about srk")

result

{'setup': 'Why did Shah Rukh Khan bring a ladder to the movie set?',
 'punchline': 'Because he wanted to take his acting to the next level! Jhooom!'}

# ** Chaings IN LangChain

# *First Chain

In [34]:
prompt_template= ChatPromptTemplate.from_messages([
("system" , "You are a nice assistant"),
("human", "{input}")
])

In [35]:
#task2

llm_claude2 = init_chat_model(model ="claude-haiku-4-5" )

In [36]:
#task3
from langchain_core.output_parsers import StrOutputParser
str_parser = StrOutputParser()


# ** Manual Invocation

In [37]:
template = prompt_template.invoke({"input" : "What is the capital of India"})
res = llm_claude2.invoke(template)
final_result = res.content
print(final_result)

The capital of India is **New Delhi**.

New Delhi is located in the northern part of India and serves as the seat of the Indian government. It's part of the larger Delhi metropolitan area and is home to important government institutions including the Parliament of India and the President's residence.


In [39]:
chain = prompt_template | llm_claude2 | str_parser
chain.invoke({"input": "What is the capitol of India"})

"The capital of India is **New Delhi**.\n\nNew Delhi is located in the northern part of India and has been the capital since 1931. It's part of the larger Delhi metropolitan area and serves as the seat of the Indian government."

In [42]:
from langchain_core.runnables import RunnableSequence

chain2 = RunnableSequence(prompt_template,llm_claude2,str_parser)
chain2.invoke({"input": "What is the capitol of USA"})

'The capital of the USA is **Washington, D.C.** (Washington, District of Columbia).\n\nIt is located on the east coast of the United States and serves as the seat of the federal government, where the President, Congress, and Supreme Court are located.'

In [44]:
#task4

from langchain_core.runnables import RunnableLambda

def dictionary_maker(text: str)-> dict:
    return {"text": text}


dictionary_maker_runnable = RunnableLambda(dictionary_maker)


In [46]:
#task5

prompt_post = ChatPromptTemplate.from_messages(
    messages=[
        ("system", " Youare a social media post generator"),
        ("human", " Create a post for the following text for Linkedin: {text}")
    ]
    
)

In [49]:
#task6

llm_claude3 = init_chat_model(model ="claude-haiku-4-5" ,temperature = 0)

In [50]:
str_parser = StrOutputParser()

In [52]:
#Chain

chain = prompt_template | llm_claude2 | str_parser | dictionary_maker_runnable | prompt_post | llm_claude3 | str_parser

chain.invoke("What is the capital of UAE")

"# LinkedIn Post\n\n🇦🇪 **Did you know?** Abu Dhabi is the capital of the United Arab Emirates and serves as the nation's political and administrative heart.\n\nWhile Dubai often steals the spotlight, Abu Dhabi is actually the **largest city by area** and the **second-largest by population**. Strategically positioned on the Persian Gulf, this dynamic city is the true powerhouse behind the UAE's governance and development.\n\nWhether you're doing business in the region or simply expanding your geographic knowledge, understanding the role of Abu Dhabi is essential to understanding the UAE. 💼\n\n**What's your experience with Abu Dhabi?** Share your thoughts in the comments below! 👇\n\n#UAE #AbuDhabi #MiddleEast #BusinessInsights #Geography\n\n---\n\n*Feel free to adjust the tone or add specific details relevant to your industry or audience!*"